# Cell 01 - Tải và kiểm tra cấu trúc dataset Vietnamese Legal Documents

In [2]:
from pathlib import Path
from datasets import load_dataset

cwd = Path.cwd().resolve()
PROJECT = cwd.parent if cwd.name == "notebooks" else cwd

RAW_DIR = PROJECT / "data" / "raw"
PROCESSED_DIR = PROJECT / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

DATASET_NAME = "YuITC/Vietnamese-Legal-Documents"

ds = load_dataset(DATASET_NAME)

print("Project:", PROJECT)
print("Dataset:", DATASET_NAME)
print("Splits:", list(ds.keys()))

for split in ds:
    print("\nSplit:", split)
    print("Rows:", len(ds[split]))
    print("Columns:", ds[split].column_names)

README.md: 0.00B [00:00, ?B/s]

C:\Users\npd20\AppData\Roaming\Python\Python311\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\npd20\.cache\huggingface\hub\datasets--YuITC--Vietnamese-Legal-Documents. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


train.parquet:   0%|          | 0.00/64.5M [00:00<?, ?B/s]

test.parquet:   0%|          | 0.00/21.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/89261 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/29746 [00:00<?, ? examples/s]

Project: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag
Dataset: YuITC/Vietnamese-Legal-Documents
Splits: ['train', 'test']

Split: train
Rows: 89261
Columns: ['question', 'context_list', 'qid', 'cid']

Split: test
Rows: 29746
Columns: ['question', 'context_list', 'qid', 'cid']


# Cell 02 - Kiểm tra mẫu dữ liệu train/test

In [3]:
import pandas as pd

train_df = ds["train"].to_pandas()
test_df = ds["test"].to_pandas()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain columns:")
print(train_df.columns.tolist())

print("\nKiểu dữ liệu context_list:")
print(type(train_df.loc[0, "context_list"]))

print("\nMột mẫu dữ liệu:")
sample = train_df.iloc[0]
print("qid:", sample["qid"])
print("cid:", sample["cid"])
print("question:", sample["question"])
print("context_list:", sample["context_list"][:2] if isinstance(sample["context_list"], list) else sample["context_list"])

display(train_df.head())

Train shape: (89261, 4)
Test shape: (29746, 4)

Train columns:
['question', 'context_list', 'qid', 'cid']

Kiểu dữ liệu context_list:
<class 'numpy.ndarray'>

Một mẫu dữ liệu:
qid: 72600
cid: [142820]
question: Liên đoàn Luật sư Việt Nam là tổ chức xã hội – nghề nghiệp có tư cách pháp nhân, có con dấu, tài khoản riêng?
context_list: ['“Điều 2. Địa vị pháp lý của Liên đoàn Luật sư Việt Nam\n1. Liên đoàn Luật sư Việt Nam là tổ chức xã hội - nghề nghiệp thống nhất trong toàn quốc của các Đoàn Luật sư, các luật sư Việt Nam; có tư cách pháp nhân, có con dấu, tài khoản.\n2. Biểu tượng của Liên đoàn Luật sư Việt Nam là hình tròn nền xanh da trời, chính giữa là cán cân công lý gắn với hình tượng cuốn sách, dưới cán cân công lý là dòng chữ “VIETNAM BAR FEDERATION", hai bên mỗi bên có ba dải màu vàng đậm, phía trên là ngôi sao vàng hình cờ Tổ quốc Việt Nam và dòng chữ Liên đoàn Luật sư Việt Nam.\n3. Tên giao dịch quốc tế của Liên đoàn Luật sư Việt Nam là Vietnam Bar Federation (viết tắt là VBF).

,question,context_list,qid,cid
0,Liên đoàn Luật sư Việt Nam là tổ chức xã hội –...,[“Điều 2. Địa vị pháp lý của Liên đoàn Luật sư...,72600,[142820]
1,Tên hợp tác xã bị rơi vào trường hợp cấm thì c...,"[""Điều 7. Tên hợp tác xã, liên hiệp hợp tác xã...",147562,"[27817, 72117]"
2,Tài xế lái xe ô tô khách 50 chỗ ngồi bao lâu t...,"[""1. Sử dụng lái xe bảo đảm sức khỏe theo tiêu...",142107,"[33215, 56201]"
3,Các bước chuẩn bị thủ thuật bó bột Cravate sẽ ...,[BỘT CRAVATE\n...\nIV. CHUẨN BỊ\n1. Người thực...,77353,[148158]
4,Viên chức Hộ sinh hạng 4 có những nhiệm vụ gì ...,[Hộ sinh hạng IV - Mã số: V.08.06.16\n1. Nhiệm...,113090,[188132]


# Cell 03 - Thống kê số lượng context và độ dài tài liệu

In [4]:
import numpy as np

def add_stats(df):
    out = df.copy()
    out["num_contexts"] = out["context_list"].apply(len)
    out["num_cids"] = out["cid"].apply(len)
    out["total_context_chars"] = out["context_list"].apply(lambda xs: sum(len(str(x)) for x in xs))
    out["first_context_chars"] = out["context_list"].apply(lambda xs: len(str(xs[0])) if len(xs) else 0)
    return out

train_df = add_stats(train_df)
test_df = add_stats(test_df)

print("Train num_contexts:")
display(train_df["num_contexts"].describe())

print("Test num_contexts:")
display(test_df["num_contexts"].describe())

print("Train mismatch cid/context:", (train_df["num_contexts"] != train_df["num_cids"]).sum())
print("Test mismatch cid/context:", (test_df["num_contexts"] != test_df["num_cids"]).sum())

display(train_df[["question", "num_contexts", "num_cids", "total_context_chars", "first_context_chars"]].head())

Train num_contexts:


count    89261.000000
mean         1.116927
std          0.369105
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          9.000000
Name: num_contexts, dtype: float64

Test num_contexts:


count    29746.000000
mean         1.119445
std          0.365177
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          6.000000
Name: num_contexts, dtype: float64

Train mismatch cid/context: 0
Test mismatch cid/context: 0


,question,num_contexts,num_cids,total_context_chars,first_context_chars
0,Liên đoàn Luật sư Việt Nam là tổ chức xã hội –...,1,1,767,767
1,Tên hợp tác xã bị rơi vào trường hợp cấm thì c...,2,2,2218,1407
2,Tài xế lái xe ô tô khách 50 chỗ ngồi bao lâu t...,2,2,2258,394
3,Các bước chuẩn bị thủ thuật bó bột Cravate sẽ ...,1,1,1429,1429
4,Viên chức Hộ sinh hạng 4 có những nhiệm vụ gì ...,1,1,1921,1921


# Cell 04 - Tạo legal corpus từ context_list và cid

In [5]:
import pandas as pd

def build_legal_corpus(*dfs):
    rows = []
    for df in dfs:
        for _, row in df.iterrows():
            for cid, ctx in zip(row["cid"], row["context_list"]):
                rows.append({
                    "cid": int(cid),
                    "context_text": str(ctx),
                    "source_type": "legal"
                })
    return pd.DataFrame(rows).drop_duplicates("cid").reset_index(drop=True)

legal_corpus_df = build_legal_corpus(train_df, test_df)

out_path = PROCESSED_DIR / "legal_corpus.csv"
legal_corpus_df.to_csv(out_path, index=False, encoding="utf-8-sig")

print("Legal corpus shape:", legal_corpus_df.shape)
print("Unique cid:", legal_corpus_df["cid"].nunique())
print("Đã lưu:", out_path)

display(legal_corpus_df.head())

Legal corpus shape: (68663, 3)
Unique cid: 68663
Đã lưu: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\data\processed\legal_corpus.csv


,cid,context_text,source_type
0,142820,“Điều 2. Địa vị pháp lý của Liên đoàn Luật sư ...,legal
1,27817,"""Điều 7. Tên hợp tác xã, liên hiệp hợp tác xã\...",legal
2,72117,"Cơ quan đăng ký hợp tác xã\n1. Khi thành lập, ...",legal
3,33215,"""1. Sử dụng lái xe bảo đảm sức khỏe theo tiêu ...",legal
4,56201,“Điều 21. Khám sức khỏe và điều trị bệnh nghề ...,legal


Giải thích ngắn gọn cell này làm gì

Cell này lấy toàn bộ context_list và cid từ train/test, tách thành từng tài liệu riêng, loại trùng, rồi lưu thành legal_corpus.csv.

# Cell 05 - Lưu tập câu hỏi và ground truth cid

In [6]:
import json

def build_queries_df(df):
    out = df[["qid", "question", "cid"]].copy()
    out["relevant_cids"] = out["cid"].apply(lambda xs: json.dumps([int(x) for x in xs], ensure_ascii=False))
    out["num_relevant"] = out["cid"].apply(len)
    return out.drop(columns=["cid"]).reset_index(drop=True)

train_queries_df = build_queries_df(train_df)
test_queries_df = build_queries_df(test_df)

train_path = PROCESSED_DIR / "train_queries.csv"
test_path = PROCESSED_DIR / "test_queries.csv"

train_queries_df.to_csv(train_path, index=False, encoding="utf-8-sig")
test_queries_df.to_csv(test_path, index=False, encoding="utf-8-sig")

print("Train queries:", train_queries_df.shape)
print("Test queries:", test_queries_df.shape)
print("Đã lưu:", train_path)
print("Đã lưu:", test_path)

display(train_queries_df.head())

Train queries: (89261, 4)
Test queries: (29746, 4)
Đã lưu: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\data\processed\train_queries.csv
Đã lưu: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\data\processed\test_queries.csv


,qid,question,relevant_cids,num_relevant
0,72600,Liên đoàn Luật sư Việt Nam là tổ chức xã hội –...,[142820],1
1,147562,Tên hợp tác xã bị rơi vào trường hợp cấm thì c...,"[27817, 72117]",2
2,142107,Tài xế lái xe ô tô khách 50 chỗ ngồi bao lâu t...,"[33215, 56201]",2
3,77353,Các bước chuẩn bị thủ thuật bó bột Cravate sẽ ...,[148158],1
4,113090,Viên chức Hộ sinh hạng 4 có những nhiệm vụ gì ...,[188132],1


# Cell 06 - Tải 37signals Employee Handbook

In [7]:
from pathlib import Path
import urllib.request

cwd = Path.cwd().resolve()
PROJECT = cwd.parent if cwd.name == "notebooks" else cwd

RAW_DIR = PROJECT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

url = "https://codeload.github.com/basecamp/handbook/zip/refs/heads/master"
zip_path = RAW_DIR / "37signals_handbook.zip"
tmp_path = RAW_DIR / "37signals_handbook.tmp"

with urllib.request.urlopen(url, timeout=60) as response:
    downloaded = 0
    with open(tmp_path, "wb") as f:
        while True:
            chunk = response.read(1024 * 1024)
            if not chunk:
                break
            f.write(chunk)
            downloaded += len(chunk)
            print("Đã tải MB:", round(downloaded / 1024 / 1024, 2))

tmp_path.replace(zip_path)

print("Hoàn tất:", zip_path)
print("Dung lượng MB:", round(zip_path.stat().st_size / 1024 / 1024, 2))

Đã tải MB: 0.04
Hoàn tất: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\data\raw\37signals_handbook.zip
Dung lượng MB: 0.04


# Cell 07 - Giải nén và kiểm tra cấu trúc 37signals Employee Handbook

## Mục tiêu cell này
Giải nén file `37signals_handbook.zip` và kiểm tra bên trong có những file tài liệu nào.

## Giải thích code
Code sẽ giải nén file zip vào thư mục `data/raw/37signals_handbook`, sau đó liệt kê các file tìm thấy.  
Mục tiêu là xác định corpus nội bộ công ty có dạng `.md`, `.txt`, `.html` hay dạng khác.

## Output mong đợi
Bạn cần thấy số lượng file được tìm thấy và danh sách một số file đầu tiên trong handbook.

In [8]:
from pathlib import Path
import zipfile

cwd = Path.cwd().resolve()
PROJECT = cwd.parent if cwd.name == "notebooks" else cwd

RAW_DIR = PROJECT / "data" / "raw"
zip_path = RAW_DIR / "37signals_handbook.zip"
extract_dir = RAW_DIR / "37signals_handbook"

if not extract_dir.exists():
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_dir)

files = [p for p in extract_dir.rglob("*") if p.is_file()]

print("Extract folder:", extract_dir)
print("Số file:", len(files))

for p in files[:30]:
    print(p.relative_to(extract_dir))

Extract folder: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\data\raw\37signals_handbook
Số file: 16
handbook-master\benefits-and-perks.md
handbook-master\getting-started.md
handbook-master\how-we-work.md
handbook-master\making-a-career.md
handbook-master\managing-work-devices.md
handbook-master\moonlighting.md
handbook-master\our-internal-systems.md
handbook-master\our-rituals.md
handbook-master\README.md
handbook-master\severance.md
handbook-master\stateFMLA.md
handbook-master\titles-for-designers.md
handbook-master\titles-for-ops.md
handbook-master\titles-for-programmers.md
handbook-master\titles-for-QA.md
handbook-master\titles-for-support.md


# Cell 08 - Đọc 37signals Handbook thành bảng tài liệu nội bộ

## Mục tiêu cell này
Đọc toàn bộ file `.md` trong 37signals Employee Handbook và chuyển thành bảng `company_handbook_documents.csv`.

## Giải thích code
Code sẽ quét các file Markdown, đọc nội dung từng file, tạo các cột `doc_id`, `title`, `source_path`, `text`, `text_length`, `source_type`.  
File kết quả sẽ được lưu vào `data/processed/company_handbook_documents.csv` để dùng cho bước chunking và retrieval sau này.

## Output mong đợi
Bạn cần thấy bảng có 16 dòng, mỗi dòng tương ứng một tài liệu handbook nội bộ.

In [9]:
import pandas as pd
from pathlib import Path

handbook_root = RAW_DIR / "37signals_handbook"

md_files = sorted(handbook_root.rglob("*.md"))

rows = []

for i, path in enumerate(md_files):
    text = path.read_text(encoding="utf-8", errors="ignore").strip()
    rows.append({
        "doc_id": f"company_{i:04d}",
        "title": path.stem.replace("-", " ").replace("_", " ").title(),
        "source_path": str(path.relative_to(handbook_root)),
        "text": text,
        "text_length": len(text),
        "source_type": "company_handbook"
    })

company_docs_df = pd.DataFrame(rows)

out_path = PROCESSED_DIR / "company_handbook_documents.csv"
company_docs_df.to_csv(out_path, index=False, encoding="utf-8-sig")

print("Company handbook shape:", company_docs_df.shape)
print("Đã lưu:", out_path)

display(company_docs_df[["doc_id", "title", "source_path", "text_length", "source_type"]])
print("\nPreview:")
print(company_docs_df.loc[0, "text"][:1000])

Company handbook shape: (16, 6)
Đã lưu: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\data\processed\company_handbook_documents.csv


,doc_id,title,source_path,text_length,source_type
0,company_0000,Benefits And Perks,handbook-master\benefits-and-perks.md,13647,company_handbook
1,company_0001,Getting Started,handbook-master\getting-started.md,3147,company_handbook
2,company_0002,How We Work,handbook-master\how-we-work.md,6551,company_handbook
3,company_0003,Making A Career,handbook-master\making-a-career.md,6041,company_handbook
4,company_0004,Managing Work Devices,handbook-master\managing-work-devices.md,2937,company_handbook
5,company_0005,Moonlighting,handbook-master\moonlighting.md,3486,company_handbook
6,company_0006,Our Internal Systems,handbook-master\our-internal-systems.md,2829,company_handbook
7,company_0007,Our Rituals,handbook-master\our-rituals.md,1845,company_handbook
8,company_0008,Readme,handbook-master\README.md,2161,company_handbook
9,company_0009,Severance,handbook-master\severance.md,1074,company_handbook



Preview:
# Benefits & Perks

## Health Insurance

Detailed information about all 37signals insurance policies and other benefits can be found in [Basecamp](https://3.basecamp.com/2914079/buckets/28168307/vaults/5060979274).

### Medical Insurance

In the United States, medical insurance is provided through Blue Cross Blue Shield PPO. The company pays 75% of the premium and the employee pays the other 25%. Open enrollment is in November every year, with new coverage beginning December 1. Marriages and domestic partnerships are covered. You’re eligible for coverage on your first day of employment. If you are terminated or resign from 37signals, your coverage will end on the last day of the month of your separation date and you may be eligible for continued coverage after that (COBRA).

Each pay period, you’ll see a payroll deduction for medical insurance:

* Employee-only medical coverage: $96.05
* Employee-partner medical coverage: $197.27
* Employee-child(ren) medical coverage: $186.4

# Cell 09 - Tổng kết dữ liệu đã chuẩn bị

## Mục tiêu cell này
Tạo bảng tổng kết các nguồn dữ liệu đã có trong project.

## Giải thích code
Code sẽ đọc số lượng tài liệu pháp luật, số câu hỏi train/test và số tài liệu handbook nội bộ.  
Sau đó tạo file `dataset_summary.csv` để lưu lại thông tin tổng quan của dataset.

## Output mong đợi
Bạn cần thấy bảng gồm các nguồn dữ liệu:
- legal_corpus
- train_queries
- test_queries
- company_handbook

In [10]:
summary_rows = [
    {
        "dataset_part": "legal_corpus",
        "description": "Kho văn bản pháp luật tiếng Việt dùng cho retrieval",
        "rows": len(legal_corpus_df),
        "language": "Vietnamese",
        "file": "data/processed/legal_corpus.csv"
    },
    {
        "dataset_part": "train_queries",
        "description": "Câu hỏi train và relevant cid để đánh giá retrieval",
        "rows": len(train_queries_df),
        "language": "Vietnamese",
        "file": "data/processed/train_queries.csv"
    },
    {
        "dataset_part": "test_queries",
        "description": "Câu hỏi test và relevant cid để đánh giá retrieval",
        "rows": len(test_queries_df),
        "language": "Vietnamese",
        "file": "data/processed/test_queries.csv"
    },
    {
        "dataset_part": "company_handbook",
        "description": "Tài liệu nội bộ công ty công khai từ 37signals Employee Handbook",
        "rows": len(company_docs_df),
        "language": "English",
        "file": "data/processed/company_handbook_documents.csv"
    }
]

dataset_summary_df = pd.DataFrame(summary_rows)

out_path = PROCESSED_DIR / "dataset_summary.csv"
dataset_summary_df.to_csv(out_path, index=False, encoding="utf-8-sig")

print("Đã lưu:", out_path)
display(dataset_summary_df)

Đã lưu: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\data\processed\dataset_summary.csv


,dataset_part,description,rows,language,file
0,legal_corpus,Kho văn bản pháp luật tiếng Việt dùng cho retr...,68663,Vietnamese,data/processed/legal_corpus.csv
1,train_queries,Câu hỏi train và relevant cid để đánh giá retr...,89261,Vietnamese,data/processed/train_queries.csv
2,test_queries,Câu hỏi test và relevant cid để đánh giá retri...,29746,Vietnamese,data/processed/test_queries.csv
3,company_handbook,Tài liệu nội bộ công ty công khai từ 37signals...,16,English,data/processed/company_handbook_documents.csv


# Cell 10 - Kiểm tra cuối các file dữ liệu đã chuẩn bị

## Mục tiêu cell này
Kiểm tra toàn bộ file dữ liệu đầu ra của notebook 01 đã được lưu đúng chưa.

## Giải thích code
Code sẽ kiểm tra sự tồn tại của các file:
- `legal_corpus.csv`
- `train_queries.csv`
- `test_queries.csv`
- `company_handbook_documents.csv`
- `dataset_summary.csv`

Sau đó in ra trạng thái `OK` hoặc `MISSING`.

## Output mong đợi
Tất cả file cần có trạng thái `OK`.

In [11]:
from pathlib import Path
import pandas as pd

required_files = [
    PROCESSED_DIR / "legal_corpus.csv",
    PROCESSED_DIR / "train_queries.csv",
    PROCESSED_DIR / "test_queries.csv",
    PROCESSED_DIR / "company_handbook_documents.csv",
    PROCESSED_DIR / "dataset_summary.csv"
]

check_rows = []

for path in required_files:
    check_rows.append({
        "file": str(path.relative_to(PROJECT)),
        "status": "OK" if path.exists() else "MISSING",
        "size_mb": round(path.stat().st_size / 1024 / 1024, 2) if path.exists() else 0
    })

check_df = pd.DataFrame(check_rows)

display(check_df)

print("Tổng file OK:", (check_df["status"] == "OK").sum(), "/", len(check_df))

,file,status,size_mb
0,data\processed\legal_corpus.csv,OK,88.01
1,data\processed\train_queries.csv,OK,12.08
2,data\processed\test_queries.csv,OK,4.03
3,data\processed\company_handbook_documents.csv,OK,0.10
4,data\processed\dataset_summary.csv,OK,0.00


Tổng file OK: 5 / 5
